In [1]:
import subprocess
import sys

COLAB = "google.colab" in sys.modules


def _install(package):
    if COLAB:
        ans = input(f"Install { package }? [y/n]:")
        if ans.lower() in ["y", "yes"]:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "--quiet", package]
            )
            print(f"{ package } installed!")


def _colab_install_missing_deps(deps):
    import importlib

    for dep in deps:
        if importlib.util.find_spec(dep) is None:
            if dep == "iris":
                dep = "scitools-iris"
            _install(dep)


deps = ["erddapy"]
_colab_install_missing_deps(deps)

# Exploring ERDDAP latest features


Created: 2026-10-10


ERDDAP 2.30.0 introduced some interesting new features:

- A metrics response to help ERDDAP maintainers and users get information on the server itself;
- A croissant metadata spec response to facilitate dataset interoperability with Machine Learning (ML);
- Support for the table optimized `parquet` format;
- Message Queuing Telemetry Transport Technical (MQTT) for interaction with IoT;


https://docs.mlcommons.org/croissant/docs/croissant-spec-1.0.html

https://mqtt.org/faq/


Let's start by asking the server version to be sure we have one that has the new features

In [2]:
import requests
import pandas as pd
from prometheus_client.parser import text_string_to_metric_families


def metrics_erddap_build_info(url):
    metrics = f"{url}/metrics"
    r = requests.get(metrics, allow_redirects=True, timeout=10)
    r.raise_for_status()

    for family in text_string_to_metric_families(r.text):
        if family.name == "ERDDAP_build_info":
            data = family.samples[0].labels
    deployment_info = data["deployment_info"]
    version = data["version_full"]
    return deployment_info, version


server = "https://gliders.ioos.us/erddap"

metrics_erddap_build_info(server)

('docker', '2.30.0')

Great! We got a docker deployment of the minimun required version. Supposed you are training a ML model for QA/QC of glider data and your model requires the data to be `croissant` spec compliant.

In [3]:
from erddapy import ERDDAP
from erddapy.core.url import urlopen

e = ERDDAP(server=server, protocol="tabledap")

e.dataset_id = "amelia-20180501T0000"
e.response = "croissant"

In [4]:
import json

jsonld = urlopen(e.get_download_url())
metadata = json.loads(jsonld.read().decode())

The metadata is rich and hard to read as a human, let's check if it passes croissant validation to ensure it can be fed in ML workflows.

In [5]:
import json

with open("metadata.json", "w") as f:
    content = json.dumps(metadata, indent=2)
    f.write(content)

When loading with the `mlcroissant` we can check it for compliance explorer this format a bit further.

In [6]:
import mlcroissant

dataset = mlcroissant.Dataset(jsonld="metadata.json")

  -  [Metadata(amelia-20180501T0000)] Property "http://mlcommons.org/croissant/citeAs" is recommended, but does not exist.
  -  [Metadata(amelia-20180501T0000)] Property "https://schema.org/version" is recommended, but does not exist.


There are some warnings, but no errors and the dataset is loaded. Let's check what can we do with it. We can access them again via the `metada.issues` object.

In [7]:
print(dataset.metadata.issues.report())

Found the following 2 warning(s) during the validation:
  -  [Metadata(amelia-20180501T0000)] Property "http://mlcommons.org/croissant/citeAs" is recommended, but does not exist.
  -  [Metadata(amelia-20180501T0000)] Property "https://schema.org/version" is recommended, but does not exist.


ERDDAP's `croissant` is metadata only. We need to attach the data whe passing it to a ML workflow. Implementing a ML workflow from here should be easier than writing your own data loaders/converters, etc.


Now, imagine you  do have a ML worklfow that downloads many files. While setting a ML workflow is beyond the scope of this notebook, we can show you how to quickly obtain `parquet` files that are lightweight and easy to load in many ML workflows.


Below let's compare the two most common formats for tables: `parquet` and `csv` for file size, reading efficience, etc.

In [8]:
%%time
csv = e.download_file(file_type="csv")

CPU times: user 124 ms, sys: 53.5 ms, total: 178 ms
Wall time: 5.65 s


In [9]:
%%time
parquet = e.download_file(file_type="parquet")

CPU times: user 10.8 ms, sys: 4.13 ms, total: 14.9 ms
Wall time: 7 s


The download times are similar, with the `parquet` format just a bit slower. However, it is hard to separate network time from the ERDDAP's internal time to pre-process the files, maybe creating the parquet file on the server-side here is the culprit for the extra time. Let's check the file size to be sure.

In [10]:
from humanize import naturalsize

print(f"parquet: {naturalsize(parquet.stat().st_size)}")
print(f"csv: {naturalsize(csv.stat().st_size)}")

parquet: 1.9 MB
csv: 13.7 MB


The parquet file is an order of magnitude lower! The difference can be huge when dealing with large datasets or slow internet connection. Let's check the reading times now.

In [11]:
%%time

df = pd.read_csv(csv)

CPU times: user 248 ms, sys: 30 ms, total: 278 ms
Wall time: 278 ms


<timed exec>:1: DtypeWarning: Columns (4: latitude, 5: longitude, 6: depth, 7: conductivity, 9: density, 14: lat_uv, 17: lon_uv, 20: precise_lat, 21: precise_lon, 23: pressure, 64: temperature, 67: time_uv, 69: u, 71: v) have mixed types. Specify dtype option on import or set low_memory=False.


In [12]:
%%time

df = pd.read_parquet(parquet)

CPU times: user 69.7 ms, sys: 241 ms, total: 311 ms
Wall time: 79.8 ms


In [13]:
df

,trajectory,wmo_id,profile_id,time,latitude,longitude,depth,conductivity,conductivity_qc,density,...,salinity_qc,temperature,temperature_qc,time_qc,time_uv,time_uv_qc,u,u_qc,v,v_qc
0,amelia-20180501T0000,4801934,1,2018-05-01 14:58:48.317000+00:00,37.539598,-74.856492,25.350000,3.44320,NaN,1025.799805,...,NaN,8.0939,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,amelia-20180501T0000,4801934,1,2018-05-01 14:58:48.317000+00:00,37.539598,-74.856492,23.950001,3.44254,NaN,1025.776001,...,NaN,8.1080,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,amelia-20180501T0000,4801934,1,2018-05-01 14:58:48.317000+00:00,37.539598,-74.856492,22.219999,3.46240,NaN,1025.745361,...,NaN,8.3210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,amelia-20180501T0000,4801934,1,2018-05-01 14:58:48.317000+00:00,37.539598,-74.856492,20.860001,3.48436,NaN,1025.682251,...,NaN,8.5911,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,amelia-20180501T0000,4801934,1,2018-05-01 14:58:48.317000+00:00,37.539598,-74.856492,19.490000,3.55269,NaN,1025.579468,...,NaN,9.3318,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35876,amelia-20180501T0000,4801934,1397,2018-05-14 12:29:12.947000+00:00,37.475571,-75.079579,11.010000,3.69139,NaN,1024.308472,...,NaN,12.0024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35877,amelia-20180501T0000,4801934,1397,2018-05-14 12:29:12.947000+00:00,37.475571,-75.079579,8.240000,3.88118,NaN,1023.760315,...,NaN,14.2628,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35878,amelia-20180501T0000,4801934,1397,2018-05-14 12:29:12.947000+00:00,37.475571,-75.079579,6.290000,3.88696,NaN,1023.779907,...,NaN,14.2783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35879,amelia-20180501T0000,4801934,1397,2018-05-14 12:29:12.947000+00:00,37.475571,-75.079579,3.710000,3.89055,NaN,1023.776733,...,NaN,14.2993,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


It is no surprise that the optimized table binary format (`parquet`) will load much faster.

With the `croissant` metadata and the `parquet` files one can set up tensoflow/pytorch ML workflows. The last missing piece would be a workflow that is trained on near real time data. That is where MQTT could play a role. With MQTT datasets can be updated remotely, directly from the instrument acquiring the data, and a ML workflow could re-run when the ERDDAP reports that a new data just arrived.